# 004 · Chirp pulse sweep

**Level:** basic · **Topics:** Gen2, dsl, pulses, sweeps, chunking · **Type:** write the code

Write an experiment that sweeps two parameters of a **custom pulse** and measures after each pulse.

1. Write a chirp pulse: a pulse whose frequency moves linearly during the pulse.
2. Sweep its **length** and its **frequency span** in a 2D real-time sweep, and measure the qubit after each pulse.
3. The full sweep does not fit in the memory of the instrument. Make it fit, without removing sweep points.

Cells with a `TODO` are yours to complete. Everything runs in emulation, no instrument needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from laboneq.simple import *

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Step 1: the chirp pulse

A chirp of length $T$ and span $\Delta f$ has the complex envelope

$$ e^{\,i\,2\pi\left(-\frac{\Delta f}{2}\,t \;+\; \frac{\Delta f}{2T}\,t^2\right)} \qquad 0 \le t \le T $$

Its frequency goes from $-\Delta f/2$ to $+\Delta f/2$ around the carrier.

A pulse functional receives `x`, which runs from −1 to 1 over the pulse, and the pulse `length`.

📖 [Pulse library and custom pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/01_pulse_library.html)

In [ ]:
@pulse_library.register_pulse_functional
def chirp(x, length, span=10e6, **_):
    t = (x + 1) / 2 * length  # time, from 0 to length
    # TODO: return the complex envelope of the chirp
    return np.ones_like(x)

Check the pulse. The oscillation should be fast at both ends and slow in the middle.

In [ ]:
test_pulse = chirp(length=1e-6, span=20e6)
time, samples = test_pulse.generate_sampled_pulse()

plt.plot(time * 1e6, samples.real, label="I")
plt.plot(time * 1e6, samples.imag, label="Q")
plt.xlabel("time (µs)")
plt.legend()
plt.show()

## Step 2: the experiment

For every length and span: play the chirp on `drive`, then measure.

- 21 lengths × 11 spans, already defined below.
- `dsl.play` can override the `length` of a pulse and its `pulse_parameters`.

📖 [Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html) ·
[Pulse commands](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/00_pulse_commands.html) ·
[Section tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/00_section_tutorial.html)

In [ ]:
chirp_pulse = chirp(length=100e-9)
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

lengths = LinearSweepParameter(start=100e-9, stop=1e-6, count=21)
spans = LinearSweepParameter(start=1e6, stop=50e6, count=11)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def chirp_sweep():
    with dsl.acquire_loop_rt(count=16):
        # TODO: sweep `lengths` (outer sweep) and `spans` (inner sweep)
        # TODO: play `chirp_pulse` on "drive" with the swept length and span
        # TODO: make sure the readout starts after the chirp
        with dsl.section(name="readout"):
            dsl.measure(
                measure_signal="measure",
                acquire_signal="acquire",
                measure_pulse=readout_pulse,
                integration_kernel=readout_pulse,
                handle="result",
                reset_delay=1e-6,
            )


experiment = chirp_sweep()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Step 3: compile and run

Once steps 1 and 2 are complete, the compilation fails with `resource limitation exceeded`.
Read the whole message, then make the experiment fit **without removing sweep points**.

📖 [Experiment chunking](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/tutorials/01_experiment_chunking.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)

**Expected result:** `(21, 11)`

## Questions

1. How many different waveforms does the sweep need? Why do they not fit?
2. Which sweep do you chunk, and what is the smallest `chunk_count` that works?
3. Would a 2D sweep of the pulse *amplitude* and *phase* have the same problem?

<details>
<summary>Hint 1</summary>

`dsl.play("drive", chirp_pulse, length=..., pulse_parameters={"span": ...})`
</details>

<details>
<summary>Hint 2</summary>

`dsl.sweep` has two options for chunking: `chunk_count` and `auto_chunking`.
</details>

The solution is in [solution.ipynb](solution.ipynb).